Parameters = things the model learns from data (like weights in linear regression).
Hyperparameters = things you set before training (like how deep a tree can grow, or how many neighbors in KNN). Tuning means finding the best combo of these.

The two main methods are 

GridSearchCV (tries every combination) and RandomizedSearchCV (tries random combinations — faster for large grids). Both use cross-validation internally so you're not overfitting to a test set.

In [ ]:
n_neighbors
Number of neighbors (k). Low k = overfit, High k = underfit. Try 3–20.

metric
Distance formula. 'euclidean' (default) or 'manhattan'.

weights
'uniform' = all equal. 'distance' = closer = more influence.

what is KNN and what can we tune?



KNN classifies a point by looking at its k nearest neighbors and taking a majority vote. The key hyperparameters are: how many neighbors (k), how to measure distance, and whether all neighbors vote equally.

In [4]:
# No training happens in KNN — it just memorizes data.
# Prediction time: for each test point, find k nearest train points.

# The 3 things you can tune:
# 1. n_neighbors (k)  — how many neighbors vote
# 2. metric           — how distance is measured
# 3. weights          — equal vote vs distance-weighted vote

Tip: Always scale your data before KNN! Use StandardScaler — distance-based models are sensitive to feature magnitudes.

In [ ]:
n_neighbors=1
            Only 1 neighbor decides → very sensitive, high variance, likely overfit
n_neighbors=20
            20 neighbors vote → smoother boundary, might underfit on complex data
metric='euclidean'
            Straight-line distance. Default. Good for continuous numerical features
metric='manhattan'
            Sum of absolute differences. Sometimes better for high-dimensional data
weights='uniform'
            All k neighbors have equal say regardless of how close they are
weights='distance'
            Closer neighbors have more influence. Usually better than uniform

KNN needs scaled data. Always split first, then fit scaler on train only. Never fit scaler on test data — that causes data leakage.

In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaler on train only → transform both
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Rule of thumb: fit() always on train only. transform() on both train and test.

In [ ]:
stratify=y
            Keeps class ratio same in train/test. Important for imbalanced datasets
fit_transform(X_train)
            Learns mean/std from train data and transforms it
transform(X_test)
            Uses the SAME mean/std from train to transform test — no leakage

GridSearchCV (tries every combination)



GridSearchCV takes a dictionary of param values, makes all combinations, and tests each using cross-validation. It automatically finds the best combo.

In [6]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}
# Total combos: 6 x 2 x 2 = 24. Each tested with cv=5.
# So 24 x 5 = 120 model fits happen internally.

model = KNeighborsClassifier()

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,                     # 5-fold cross validation
    scoring='accuracy',
    n_jobs=-1,                # use all CPU cores
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': [3, 5, 7, 9, 11, 15],
                         'weights': ['uniform', 'distance']},
             scoring='accuracy', verbose=1)

In [ ]:
cv=5
        Data split into 5 folds. Model trained on 4, validated on 1. Repeated 5 times. Average score used.
scoring='accuracy'
        Metric used to decide the 'best' model. Use 'f1' or 'roc_auc' for imbalanced data
n_jobs=-1
        Use all available CPU cores — speeds up search significantly
verbose=1
        Prints progress. Use verbose=2 for more detail during tuning

GridSearchCV is exhaustive — good when param grid is small (under 100 combos). For large grids, use RandomizedSearchCV.

After fitting, GridSearchCV gives you the best params, best score, and the best estimator ready to predict.

In [7]:
# Results
print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

# Example output:
# Best params: {'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'distance'}
# Best CV score: 0.975

# Use best model to predict on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

# See all combos ranked
results = pd.DataFrame(grid_search.cv_results_)
results[['params', 'mean_test_score', 'rank_test_score']]  .sort_values('rank_test_score').head(5)

Best params: {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'uniform'}
Best CV score: 0.9666666666666668
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.83      1.00      0.91        10
           2       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30



,params,mean_test_score,rank_test_score
11,"{'metric': 'euclidean', 'n_neighbors': 15, 'we...",0.966667,1
2,"{'metric': 'euclidean', 'n_neighbors': 5, 'wei...",0.966667,1
3,"{'metric': 'euclidean', 'n_neighbors': 5, 'wei...",0.966667,1
7,"{'metric': 'euclidean', 'n_neighbors': 9, 'wei...",0.966667,1
9,"{'metric': 'euclidean', 'n_neighbors': 11, 'we...",0.966667,1


In [ ]:
best_params_
        Dictionary of the winning hyperparameter combination
best_score_
      cross-validated score for the best combo — NOT the test set score
best_estimator_
        The actual fitted model with best params — use this to predict
cv_results_
        Full dataframe with scores for every combination tried

best_score_ is CV score on train folds. Always evaluate best_estimator_ separately on your test set to confirm.